In [1]:
import pandas as pd
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from gensim.models import Word2Vec, FastText, KeyedVectors


# Data preparation and cleaning

In [8]:
data = pd.read_csv("./data/french_english.tsv", sep='\t')
data.columns = ["id_en", "en", "id_fr", "fr"]
data.head()

,id_en,en,id_fr,fr
0,1276,Let's try something.,456963,Tentons quelque chose !
1,1277,I have to go to sleep.,373908,Je dois aller dormir.
2,1280,Today is June 18th and it is Muiriel's birthday!,3095,Aujourd'hui nous sommes le 18 juin et c'est l'...
3,1280,Today is June 18th and it is Muiriel's birthday!,696081,"Aujourd'hui c'est le 18 juin, et c'est l'anniv..."
4,1282,Muiriel is 20 now.,3097,Muiriel a 20 ans maintenant.


In [37]:
pattern_fr = re.compile(r"[a-zA-ZÀ-ÿ]+")
pattern_en = re.compile(r"[a-z]+")

In [38]:
def get_vocab(sentences: pd.Series, pattern: re.Pattern) -> pd.Index:
    """Get sorted unique vocabulary from a text column/Series"""
    tokens = (
        sentences.fillna("")
                 .str.lower()
                 .apply(lambda x: pattern.findall(x))
                 .explode()
                 .dropna()
    )
    return sorted(tokens.unique())

In [39]:
def tokenize_corpus(series: pd.Series, pattern: re.Pattern) -> list[list[str]]:
    """Tokenize corpus into list of sentences (from Series to list of tokenized sentences)."""
    sentences = (
        series.fillna("")
              .str.lower()
              .apply(lambda x: pattern.findall(x))
              .tolist()
    )
    return [s for s in sentences if len(s) > 0]

In [40]:
french_vocab = get_vocab(data["fr"], pattern_fr)
english_vocab = get_vocab(data["en"], pattern_en)
word_to_idx_fr = {word: idx for idx, word in enumerate(french_vocab)}
word_to_idx_en = {word: idx for idx, word in enumerate(english_vocab)}

In [41]:
print(f"French vocab: {len(french_vocab)} words")
print(f"English vocab: {len(english_vocab)} words")
print(english_vocab[-20:])
print(french_vocab[-20:])

French vocab: 46732 words
English vocab: 33044 words
['zoological', 'zoologist', 'zoology', 'zoom', 'zoomed', 'zoophile', 'zoos', 'zoroastrianism', 'zorro', 'zouaves', 'zsuzsi', 'zuan', 'zucchini', 'zucchinis', 'zuckerberg', 'zugzwang', 'zukertort', 'zulu', 'zurich', 'zwina']
['îliens', 'îlot', 'ï', 'ó', 'ô', 'ôta', 'ôtais', 'ôtant', 'ôte', 'ôter', 'ôterait', 'ôterez', 'ôtes', 'ôtez', 'ôté', 'ôtée', 'ôtés', 'ú', 'úto', 'ý']


In [42]:
french_sentences = tokenize_corpus(data["fr"], pattern_fr)
english_sentences = tokenize_corpus(data["en"], pattern_en)
print(tokenize_corpus(data["fr"], pattern_fr)[:5])

[['tentons', 'quelque', 'chose'], ['je', 'dois', 'aller', 'dormir'], ['aujourd', 'hui', 'nous', 'sommes', 'le', 'juin', 'et', 'c', 'est', 'l', 'anniversaire', 'de', 'muiriel'], ['aujourd', 'hui', 'c', 'est', 'le', 'juin', 'et', 'c', 'est', 'l', 'anniversaire', 'de', 'muiriel'], ['muiriel', 'a', 'ans', 'maintenant']]


# Word Embeddings Creation

In [ ]:
def save_embeddings_vec(vocab: list[str], embeddings: np.ndarray, output_file: str):
    """Save embeddings using Gensim's KeyedVectors (MUSE compatible)"""
    
    # Create KeyedVectors object from numpy array
    kv = KeyedVectors(vector_size=embeddings.shape[1])
    kv.add_vectors(vocab, embeddings)
    
    # Save in word2vec text format (exactly what MUSE expects)
    kv.save_word2vec_format(output_file, binary=False)

#### One Hot Encoding word vectors

In [ ]:
def create_onehot_embeddings(vocab: list[str], output_file: str):
    """Create one-hot encoded word vectors."""
        
    vocab_size = len(vocab)
    onehot_matrix = np.eye(vocab_size, dtype=np.float32)
    
    save_embeddings_vec(vocab, onehot_matrix, output_file)
    return onehot_matrix

In [ ]:
create_onehot_embeddings(french_vocab, "./embeddings/french_onehot.vec")
create_onehot_embeddings(english_vocab, "./embeddings/english_onehot.vec")

#### TF-IDF word vectors

In [ ]:
def create_tfidf_embeddings(sentences: list[list[str]], vocab: list[str], output_file: str):
    """Create TF-IDF word vectors (one-hot scaled by TF-IDF scores)"""
    
    # Join sentences for TfidfVectorizer
    texts = [' '.join(sent) for sent in sentences]
    n_docs = len(texts)
    
    # Fit TF-IDF with fixed vocabulary
    vectorizer = TfidfVectorizer(vocabulary=vocab)
    tfidf_matrix = vectorizer.fit_transform(texts)  # (n_docs, vocab_size)
    
    # Transpose to get word embeddings: (vocab_size, n_docs)
    # Each word = vector of its TF-IDF scores across all documents
    word_vectors = tfidf_matrix.T.toarray().astype(np.float32)
    
    save_embeddings_vec(vocab, word_vectors, output_file)
    return word_vectors

In [ ]:
create_tfidf_embeddings(french_sentences, french_vocab, "french_tfidf.vec")
create_tfidf_embeddings(english_sentences, english_vocab, "english_tfidf.vec")